# 🎮 Un mando con el giroscopio del celular
### De la velocidad angular ω al ángulo θ usando ecuaciones diferenciales

**Análisis Numérico · Proyecto final**
Integrantes: _(escribe aquí tus nombres)_

---

### 🗺️ Ruta de la exposición
1. 🎯 Lo que imaginamos vs. lo que logramos
2. 📡 ¿Cómo llegan los datos?
3. 🧭 El modelo: una EDO con valor inicial
4. 🧮 Métodos numéricos: de Euler a Heun
5. 🧹 Del sensor real al ángulo
6. 🚦 Del ángulo al comando
7. ▶️ Datos reales en vivo: ¡aquí salen los comandos!
8. 🖥️ Demo en vivo
9. ✅ Conclusiones

In [ ]:
#@title ⚙️ Preparación (ejecutar primero) { display-mode: "form" }
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.frameon": False,
})
AZUL, NARANJA, VERDE, ROJO, GRIS = "#1f77b4", "#e8710a", "#2ca02c", "#d62728", "#7f7f7f"
print("✅ Todo listo")

## 1. 🎯 Lo que imaginamos vs. lo que logramos

**La idea:** usar el celular como un mando. Si lo inclino hacia arriba, abajo, izquierda o derecha, el computador debe *entender* ese gesto y pulsar la flecha correspondiente de un juego.

Pero el computador no entiende "inclinar": entiende **números**. Todo el proyecto consiste en traducir el movimiento del celular a números, los números a un ángulo y el ángulo a una tecla.

In [ ]:
#@title 📸 Prototipo vs. resultado { display-mode: "form" }
PROTO = "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAMCAgICAgMCAgIDAwMDBAYEBAQEBAgGBgUGCQgKCgkICQkKDA8MCgsOCwkJDRENDg8QEBEQCgwSExIQEw8QEBD/2wBDAQMDAwQDBAgEBAgQCwkLEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBD/wAARCAEEANIDASIAAhEBAxEB/8QAHQAAAAcBAQEAAAAAAAAAAAAAAAIDBAUHCAYBCf/EAFsQAAEDAwIDBAUFBw8IBwkAAAECAwQABREGIQcSMRMyQVEIImFxgRQVkaHTCRcjQpOxsxY2N0VSVFVydJWytMHR0iYzYnWChZKUGCQlQ0R2ozRGR1ZkhKTh8f/EABwBAAEEAwEAAAAAAAAAAAAAAAABAgUGAwQHCP/EAD0RAAEDAgUBBQYDBwIHAAAAAAEAAgMEEQUGEiExQQcTUWFxFCIygZGxFaHRFiMkM0LB4RdSYnKCg8Lw8f/aAAwDAQACEQMRAD8A+qK1Yrm9Y6zt2jbc3crkzJdadfDADCUqVzFKlZPMQMYSfqqfeVhNVTx1c5tMxEeVwQf/AE3KEJU8etKE7W+7/kmvtK9+/tpf+Drt+Sb+0qiMeQpYDbNCdZXl9/XS/wDB92/JN/aUYccdMq6W+6/k2/tKo5KQetLAAChFgrpXx10ujrb7r8Gm/tKKePGlh+112/JN/aVSchOT7qbrG+TQiwV5/f60r/B13/JN/aUPv86V/g67fkm/tKovr+LXmAKTdPAYr1+/zpb+Drt+Sb+0off50t/B12/JN/aVRRGaFF0mgK9fv86W/g67fkm/tK9HHjSx/a67fkm/tKomhSXKd3bVe/399LfwfdvyTf2lejjppc/tfdfyTf2lUWBnxo6R40u6O7aryHHLTB/a+6/km/tKCuOWmE7m33X8k39pVIJPKaClg7+FKgxt6K8m+NumnO7Aug97bf8AjpVPGXTyjgQbl+TR/jqkYagogeQp20fXpLpNAVz/AH37B+8rj/wI/wAdD77+n/3lcfyaP8dVAVnNAL86VGhqt4cYNPn/AMFcfyaP8dKJ4waeT1hXL8m3/jqn8pPl9NAqB2ptylLGrTzTvNS4OajITpUkHNSKDtTlhR6FChQhNn+7VTccT/k7FH/16P0blWy/3aqXjj+t6N/L0fo3KEdVS6QehpQZpMZKhSyBmhPSiE7A+dKAZoqemKMOtCEi+DnIpAg43py7kg03BNCEQeVApzvR/DGK8IxvtSFOaLomDQrxROelM37xaYrpYk3SG04nGULfSlQ+BOaZZzvhWdrC74QnlCik5wQcg16DQb9UlrJRKsGjJUc70kSR0oyV0oKQhLBXsoi/I+dDmyMiknFlO5xTuU1PYZIVT5CsKxmouKvI5s08acweYGkIQnoUrOa95jSAdSfHFe84O2aQEpbJcKwaMpwK8abDY0fnpULSttPqCphvpUPbe4KmG+lOWsj0KFChCbP92ql44frfi/y9H6Nyraf7tVNxw/W/F/l6P0blCOqphPTI8qVQkjekQPqpwg7UJ6UHSjJG9FHSlmx7M0ISDowDtTc4PQU8kYxTMgb4NJdACCsAE0Qq22ox3FFOASTjApqyAWUfebtBsNpm3u6PhmHAYXJfcP4qEJKifoFfNlD83iZxNnaluQ5zOluTnhjZKM+oj3AcqfhWq/TU1+rTvD+LpCC9yS9RvFL2DuIreCv6VFCfdms6cKLYqLaHrm83hyWrCD/oJ/vOforoeRcIbWVAc8bHf5D9V1Ps5wRuJVbTKLi9z6D9StvcJ9TjUehIcuQ7zSIKfksknrzIA3PvTg/GuxR27m7cSSoHGCGjvlAWPpSQfcRWeOBF9TGvkzTL75Q1eGSGh4B5IJHxKeb6BWhfl11aVzt/JgQ4pzZnlBJHKM4PQJwn4A9d6g844T+EYtJE0e673h6H/Kgc+4KcCxuWFgsx3vN9ClAl7lQtUSQlK0KcSVtkAoBAJ9wKgPiPOvQMjyojdyuPZstOtxyEsORiopJJbWckDJ23CTn2UxevEGE6WpMtKV/uSckVVyR1VNF+qkugzmmspwpTkk0yXqS2YIS8VHwwmo+TqHtjyoZHvJphkaOqXST0XRW9alIwQfjUg2vzPWuMbvkhoZRy0uzqaQgjtEpV50hmbblKGFdiD49a9Cz+5rn42qIqwO29T21KsTGZCfwTgVtnbenNe13BQWkJ72maOHARg02bX5eNG58Hanpl1qO2dwVMN9Kh7Z3BUw30p61kehQoUITZ/u1UvHDbT8X+Xo/RuVbT/dqpeOP63o38vR+jcoR1VMpOSNqXSNulNkHenCFE+NCyAApUbClWiQrNIgmlUYyM0FL8KLJOQcUyKgDvTt85zTNQ3zTfJPAXhXttRFucu/WlNsZNIvIS4lSVZ5VDBwcH6aLW5ShfPb0idar4o8Z5sW3u9pDtzibTC5ehCFEOK+LhX8AK6mHAZt0NmFHAShltKAPcK7X0nuFPD/hlw7a1ZoXTrNruybow0mS244pQSoL5tlKI3wPCqg4ZXy6Xu3THrpKXIWh1KUqV4DHSu3dn1fSkdy1p1HYHbovQ3ZbidGf4drDqdsDtsAL/AJruLdcpVnuEW6QV9nIiuoebV/pJORWvrZqKDd7HFvzToEaWwl4KzsMjce8HI+FZ64N2rT1wvlwXqO3RZkdmCXEtvtBY5+dIyAfHBNd/NmpWhu2WeG3DtrOezjsICUjJyTgbdTn41Xe1PFKU1LKbSe8Z16EHooTtdrqaqrY6NrCJY/6trEHoumvOp3JHNHtqlISTuvoT7vKoERXnXOda1KJOSSdzScNDuwWlWfbU1HQAQfGuMyVRkK5K2MAIrMHKMEHm99OG4KgOtOEcoOaWBAGeYU0FZC0BNex5MpJBpN5KUpJ9lB93DnWmcuWAkjmHSlum2Ue9cHmJBRzDl8KkId2djrS8w4Un2eNcncZmZRSlWSadQ5B2BrCJXNNwnhgI3ViWzWjauVqcCkq/GHQV0rMxt9HOy6laT4pVmqmz0HlSsO6zLc92jLqxvuM7H4VIQ1wIs9a76f8A2r6CWzuCphvpUPbO4KmG+lSaiEehQoUITZ/u1UvHE40/F/l6P0blW0/3aqXjjj9T0bP7/R+jcoR1VLI60qg4Vk0iD5GjhQ86E9O0qBAOaMOtNUu7ADwpVpfMd6QFLqKK+sgmm3MfE0pIV62Ka8xHUUW3ShxS2TRVkDY0TmJPXego7b0126yAhUD6a4zwbRjp88Rv6LlZo4PD/sedt/36f6NaM9NiYgcLo8BJHMq6R1kE+SXKpP0Z7dAuVzRDuiQY65nrJJwFENkpB9mQK6RkmqbRA1LhcMDj9AutdnVUKF/tLhcNDj9Arb4aWS9/K37iLbIRHcj8qXVoKUr9YHbPXp4VaUCKhIBKMHG+RUwt5htkNoSlIAAwMYpk7MaSOUAVzjNOOnMGIOrXN032A8gq5mHGZMw4g6tkZpvYW8gjBpAJWABjeiCSjmwCNjTKVdo7LS1KcAKQSc1Xl01w58scZgEcoPez1qsgbqI7skK0HLtEbGFPpz5CiC9MBJ5XNsVVUW6TJCuZZ5id9smpqEt5zBcVgeWazsJPKZ3ZXUv3lK1FaVZA8cVCXO/pSkkKGPZTWS+lKVJScbVyl1uJAU2pWTTzcJvdhSce5GZPWR0Aro4ajgE1ylgilRL2TlQrqowKcBVazhYp2kKWbcyN+tKFPN4CmaFEYIpwh4Ab0iaRuvofbO4KmG+lQ9s7gqYb6VaFW0ehQoUITZ/u1UvHEZ09G/l6P0blW0/3aqXjhj9T8bP7+R+jcoR1VKgE9BQUAnxqpNecbV2u4Pae0lHblS2Vcjkhe6EK8QkDvH21SWsvSF17YLiuBK1GpT6PWWltpKEoJ8OnhWwaWQN1nYIa9pOkcrY4WB4ilWlgH1lJHtJrBaePHFjUauzs8m5yyDuY4UcH/ZG1N7nrbjuhCnJzN4abA9YqU4k7+zx+itV0kbTpc4LaZSTyC7WH6Ldk+82mKopkXWK0odQp1IP56h3NbaUaUUuait4IOCPlCf76+dkzWur21KbuEp5L4PrBxJ5gfbzb00OudQhPIZyznxAHWsgGvcFY3RSNdpIX0dja40nMdU1Hv0Na0dQHU03uPEPSVt2k3yNn9ylfMfqr51w7jrC+viNCeuTpdOwZCjnHXpTi923UtobbM9+6tPOdBJS4jI8xnrT+5uLpNLh0V6ek9qiLrrT3yWxy0SVpmNuJaScEISFZO+POqo4VXROnIkozpaIb6ZKHGytYSdh1FczZrVfL/LVEYf5lIRzqLjgQkAHqVKIFdZF4Q3Ka2H379aGUEApJlpXzErCMepnxNT+X8UqcKdaCMP52PnturnljMFVhUoMEIfa+xv12Wg9N8UW7yxyyn4vMBs628nCz/FzkGns3V7EdKlfKkjAJ3VtWajoZUNpT8C8Rpi2iolLKiBhJSFbqAG3NknpgE+FR0hyWvLK5UglKuVaFOHYjYjHsqr4nRuqqh0wjEYPQcD6rBWVplmM5i0Bx4HH5qxtd8an2luQrM4l1RVha0qPKE+8VzNl4yPQ1ETbPGk8x23KfpO+ag4tnguMgONgrPjmlDZLawMhpHMeud61mYeGKHlqZHnmwVq6a4zaTnzC1KiOwVYAyo8yM+8bj411r/EPTKW+ePNaUB13zVCRbYrGIkZxxR6BtGfzV18nTnzNaGHrjGUh99RThfht0x+c/CtluFPkaXtGwT4asPOguFwPEXXWS+JtqflIix3iVOHAUE7E++pBmBMmpMhXdHrEnpVRiDHM1g84GXEgeec9BV+SVIg2RTPTCMHw3qKqWGE2WwyTWLoabJIUkju+VdGHEHYHJrh7HcFR8pJ73jXQN3AkhVahBcnu4U6lY2GaUznfNRbMkrHMVdKdtyk43NN0lY7Er6Q2zuCphvpUPbO4KmG+lWZVxHoUKFCE2f7tUT6VF7Tp7hpJuyl8hZfHKR1Ci04Bj41ez/drKnp+zFQ+DMfl6PXyO0o56DsXz8e7T4xqcAU15sCVg5m/i1CXekrLjrbbi0E7nnPQnPtrkeF2hZHGjiQzb9QS1liOO2kqQnHMhJ7uR0yT9dSE51J07NWtIKVqbbSfaT4/AV3XokpimVqCYGsPIW02nI33Kv7hS5lqzS0nunpZSuUqJlVV6pN1pKDoq1aWYYt8C3RorDKQlttppIHKNt9tzUgbZby2FkNlWdxyjanUiQiUwyEFXOlABzRHbdObiplriuJZOMLIwDXEal8s0zjDdwHPku1wwxxMAIAWDONbEedxavDTKhyqm8gGPHYGmF31rf7Nd5tusz0aHHYeLbaWIzYISNgM4yelP9ahVy4wXNpK0oCro5uvYgBe314pd6x2dVymJfmXCWtx5RKI8InByduYjFdowGKT2JlvALkGY8SpaKqcZTuTtYXXIzNaa1muhar9PKkghIQ6pGAf4uKWvlyuc7TFoN0eefdLr6uZxZUcZSBuT7DXYN6WtbLeUWiUE9ea4XBDQ+hGSKmZsSxs2GIykw25pyWTFjqloQObfBxufbtvU62me4e8VU5MzUxB7thPr/i6rvSlsfeh3ZUdC1OGMEpCQTklY8q6mz2m+tWuCkQ3UH1SoKHL/AOJ5snPsGalYSblFgSXXJF0W0OUDtFNRU4zvjHrD41IwXIU6NDSVMrJ5dudckn8Mc7+P9h9lTmCQiGRxv0VjyZj8stVIWtAAb5+IXL6ds11RFlJdmW8KQpxRScO+qtBCgoYORsDy43x76Kzo1l5Zdel3CStRKlKailAUfElThArpIztxtlkmuRlyEIfWG+VTbUUBOOozkgfXXNG7WuOrE+523I6hxx6cr6MhNR80TC4h/iqZiGP4lWvMbXGwJsAPP0ujrsdrgnK46FK8BKnJ5v8AgaBP10+hBDJ5mILOB4tQCSP9t0io8a0tscFMV24PeSWUtxmz7uUFX103Yuzt9mojxLPFQeqnZKlyChI6n1jj6qxsbFqDWclasNBimIPDLOJPif1Vgabjm6SUESnSSrkQhUhOM+1LQxt7TTrirFah2q3xW8BXaK5cJI2AGeu/U/XTrSan4cYzHXAhvcMpQA2lKQdzttuR16d7wqu9datc1FdHJDLp+TsjsmB/oD8b4nJ+IHhW7WuZTQGIclMoaOojx4RNI0wg6iOC4jj5KIulrulk1PZIlxZDZmdhKaAVk9mtWxPl0q67vAvEq2mYxBfciIOFuoSVJHvqjNC264ap4hWO0tvqceekJSHHFFXIhOVePkAa3pbIMW229FtZYQWEICAMbYqk1tO2WTZdD9oDOFmiAns9ldRUyy+nlGDuKTuGquHetdXSdOaU+cIF8ZlPRjFkRwll5bZUFFKkkgZ5TjOKSkQp9reMecwtlxJwUrG9aEtI+nFyNkU2JU1WSyJ4JHIvupRuUvlABp01J3wCR76gkSwnGT1pZEknxrXIutvUF9UrZ3BUw30qHtncFTDfSp1VtHoUKFCE2f7tZC+6My0ReDNp58/hdTRmxjz+Syj/AGVr1/u1jX7pWEq4N6eSrOFauijb+RzKywbyC6ZJ8JWCZhUjTClEJUHJCEjfGPVUf7K6b0bXHkuxUpeDLc2/R2XSDuUgE4Ps3rkLqpULRbjy+VQXOQhO24AbWT+eum9GaSh242Nl5rmbVeFvKJ6K5W+lambbdwL8KyZIsKkrZsllLEhSGlZSCcHGM07m6guEiAITygWW0+qkADpTZ91p11a0bAnKc+FC5RWW4qih5K1FslWB0OK4oKiaKR4hNmk72XZnMY9oD+eiwVLU/L4qSJbbygXri6FKSAonKzty+PSpu9RJgkPuuxpxb5lHnl3BMZsZP7lO+PfXHRHn5PEZRjqUFLmvLTjrnKjtXL3Bd0uU1Tj7j7zm4JWpSiN/bXd8HlENEwW6D7LgeZsGkxGs1tcAAT03XdKvttthKjc7FGKeoZYVLcB/jE4zT3UWu7Q1Dt7D79zlKcjhfK24mMhQydyEbj3VWTdolr5ipKUjrlRruuIVk03bNKWB5iUXb4qO2mQ2F+qhsjIOMePvqTFS5wOlRMWWqdgJlJJ+iWZ1KpjTkm426xw2Ul9DYL5L6jso5yrqdvrqYtt5uFwg2152X2bjyWiQ2kIH/tCx4Y8BiuOtEiO3ot75fhxK52EDmIxhG/567fTFqm3G2Wp222SQ62lCFc6GypKQH3Cd+m2R9IqewFzpJHh3gr1lCiw6gmkedIOnqd+VyrVk1BP0BLvBbeeYiT+d9al91HIADv13UOlckktqVkpxWgGuG2vp+k0aYnz4FiTJfUuQzKmJSpaAkEAoTkkjGceymto9HyyyG+dd4vt4X+4tNqU22fPDz/Kj41oGgmledlz/ABTNuCUMrg2S+52aL/bb81S0dQyEp3z0FWjojTnbOtW9tJL7pDklQG7aR4e8Z281e6u1tHDTRdhfBVpqCJDSssquV2VNfUsHoWIw5duuM+VWBbnVWCAZCENQmCgLW4xBZtzQSD+6cJc8M5O+46VIUOHinJfId1A1PabDhdO+SlgJkcLNLiBa/W264bVuktYS7IYGmrE+Uvp5VOr5WW22cb+usgbjA9o5s1W0jhy/FtU6bctW2VC4jfN8mhvGY57ldlkI9+/txV0tzY2ppBltxYVzcVgDsosq8dPJbpQwk7e2ud15JhxrDPi3i7riJ5ABFdujDCyM7hMaGnr7FqI86Wpo4nXe7c+qpeDZmxPvW07LN1uBdYXO56k3VR8F9R2XTHFK23q/vJahxW3TzHqVlCkpA9uTWvla0v8AcWwuw6UUlCxlL854NjHnyDc/TWC7wu1SdRPIsLT6IYdAZ7YgrKfM49ua3rbkLZgRkKTghpGfoFaWE4ZBWOcZxwmdq+d8WyzJFDhrgNd+lyqu0rwOe09rv9Xki7gylSnpa46EcyCtzmJAUcEDKvKramQrXf2BCvMQOA7IcSMLbJ8QfH3U2fu0KNIjxX5TaHpa1IYQo7uKSCVBPngA5rpLHaUqxcHk4I/zYP56l8Sgoaajc14uP79FyTJtdmrHMyQywvLX9XWsNF97jr+qpfVvD69abeU8hCpcLPqvITuj+MPCubQ6pA9atTONtuoLbqErSQQUkZBFVbrThMmQpy4aZUG1klS4yu6f4p8K5bJT73avakU5IAcvobbO4KmG+lQ9s7gqYb6VvqMR6FChQhNn+7WNvulC+z4OafVkfrtijf8Akcytkv8AdrGv3SoE8HNPADJ/VdF/qcyssH81vqmPF2kLAWoUI/UVHWpSSpUhxWPEYbxnf3iut9Ge3PLl6dWo5BmynBjzDe1V/rN9aLJb2ieVALyicjfPL/dVp+i6tpy5WJkFSAw3Md9YYyTgZFQ2ei40vungK15FiD6kuI4WpW+ZJwo5o059DdulLUT6jKlfQKUQnnUcY61H6q54mm7nJ39SI6rp09U1w6COWRwPmF2R5BaQsF6fYku62Eu2KV8qSt5bfKnmUSQoYSB1JzU6eB3Ft9SX39PiGytAcMiVJbaQkKGfWKjnIzuMZHjUFoCXJXqtlbDcxx45VyxHQy9vt6izsk79a0NdnrE0627eWNLwJCWmxi/XVy6y88o6R0HGfcTmvTmB4fHJSt709B9l5c7Rc0V2B1jYqID37kki528rqobbwZtTckNah4m21x3PrxrKw7cXQfI8gABq2GeDGh5TrlwnaYvc1MaGjkevM1u3RF8qQAMJ/CJ9uRtU8iy3+Ta4z9ue1FPivp5m27XHi2SPy58lnt0j2nqN/GorVlz0ppFaXTqLS1rlLbCJZlpdvMxKhjACebHh4jGasnsMVK0FzLev+VyR+Z8XxfUxszifBv6NAIRbdabTabUmPpeJp1hXaqUkafs714eGw2DzmEc3tIxRFvS0pxefnDncXyL+fLs1DBXtt8njZ5lY5fVGyhyjOUmoy/8AHHSTWmYbq7hqW/8AbKcCUCQm1srxtulkZ5fIVzjWu3pDEWVYtPWu0fKG0ELQz2zoSpbvq9ovJ27PIPXKledSFBKx7nNjPA6K/dn2XMaqqmZ74yLt5cfPz3/JWjpxp+2WT5y09CYcaXzJSiw2gNHmwAV9tLVvv+N44qBn3m3F5EfU12sgeGSpF7vL1zf9pMVjlaTjr1xVK681hrG6220Jvd/mSUvxlOqSXSEKKlHB5RgeHlSVitKYEQSHk8r7yQpYxulOxSn3nYn3gedaUVR38pa0bDkrPhvZdLiNS41cwaxpJNhx8z+i0Lbdc6eT/wBSts27SojDeOxhR2bdFyepAQC7kkgYJ/FPka4riVxbl215q16dtVvhSVfhnXXGxJdQDjlTlzmHNtnp5VFtTI2l7K5NmJBLKS6tJOOd07JQPdkJ9hKj0NVPLuUi6THp0twKefWVrOMDJ8h4D2VgrKpzTpjK0aPLGGYhib5WNL4Ytm6j8R8drBdCjWeoNQ3eM1rPUk6Rbj2inm1yFoawEKIHIhSRjIGwIpzNm6H03DYVCjNz1zMOhaUlxTRSlSc8uUI2Kj1GFePrDmrkjhQ3ORSDtvkS1IYhMLeedUEoQ2nmUo+QFaILyLk3VolwGAytdD7jR0bsu79GnRNi1/rW42nUFqVOajwe2Q2h5TZC+0SlJBBGTlQAGdyelbQct6Y3I2W2UAJA5ASpSR4Zz0299Zc9Di0Pt6i1TecoR8nYYiqSUgrClOFY2PTBaG+xzjFalIwcGpjL+COqpfaah1o97Nv8VjYk+QP19Oea9o2ZKahcaOkiDp22DpCL93qFwAT/AFEC9+g434gbnpOTeL3ptyLMjxrZZX3ZD0QM+s6osqbb5FDuhIWvI8cjyruEgISEp2SBgDyqLivIZcK3EkjGBjwNO/nFj9yv6B/fTM04VXVddqpo3OZYel+Nvlb53WHs6xzBcKwzVWzMZMSRvfUW3uL89SeOlvBOtzXhH1U2+cWPBK/orz5eyrrzD4VWTl/EwL9w76Lorc6ZfcbCsZ9bLYts7gqYb6VD2zuCphvpUOp5HoUKFCE2f7tZY9PHR9w1twvs1rtpQHmdSR5GV9ABFkpOT4d6tTv92qV9IzszpSG24Mhy5Np/9J2pHCYG1NdFC7glV7NldPhmC1FXTEB7W3BPF7hfPRfoyyNWW5DE7WMWBIhKWytuPG+UhPNyn1vXRynGDjHQg133D3gseHzEARb61OkQUOI7ZcXs+cLOSMBSsD6a7eVo7SM6Z84TtMWqVKH/AH78NtxwdB3lAnwH0VKtR2GEpQww22lIwkJSAAPIYro9TknCcRj7uuh1eYe8X+hC4nS9q2Z8Md3tDVaSeR3cZA9CQSR6pe3SHoy0OysLUMcwQTj4Zo2sn06lsNytURBZcmxlsoU4rKUqUkjJxvikqFRkfZbliFuiOAgXv8b/AO5Klv8AXjPLviqmn/tx/wBmhZosfotaotc52XLu1jmNqbKexUt5CV7g4UUpyAcb4Oa6pzhlxfakFGm5uh9LxkNBttdviOOv7AAAqW2MjA69envq7aFWOLLNBA3RGCB6qq4n2g4zjMglrix5F7XaOvlx+SrDhbpnjdYbm598fWtsu9rEdYbaYyt3tipJBKlMoPLjn/GPgMY6TeuOGOi+IjjkTUem1dsGUqRdGORt1JyQEBYPMopAzhSSjcdT07PO+N9vZXtScdBEyH2d13N/4ve+/wCXgoCXG6p1Z7dDaKQDmMaPnZttz1vz1WTOLHAHUGkrDFf06hd4ttuQ4XXEIw+2kqKuZSB1AB3UnyJIArkbK881bLYCSQWGic/xpVa41zxD0nw9hm56lvQZIbV2UJvlU9IO2OVHeJ2xnISM7noRmJ2Q1qKUzcosNmAieoPoYb7jKVrlqCRgDYZx0GwqBGGU1HVOFK7cjdvNv/vgV667Csw4zmNlR+IR3DWjTJa2rfi3BI8W7dDuvZ2mYD1k01fJM9DixCSBEKemMkEny3yR4ge2nVot5mXLtilXYxFBairqp07pz547x8Og6KFRl3behsRmmw46Wm248ZA6rOwSB7ScH6D4VOybo3o3TCpLikrmEFDSjvzvr35h7B1/ipSDUFK1tEwsHPJVhzviP4JQjCaP+fMd/EDquP4l30SpiLBFP4GEeZ/B2U8R0/2Rtv4lXhiuNaSsHPSlVqLzqnVqKlrJUpR6knqa9bbUdkjODUGCZHl5UFh1CygpmwN6c+Z6o6dhuacQrnOs8xm52x4tSGFc7bg/FNFaYcXtyKzny8KNMaaaaxuNvGtgOLASFvjbdXf6GmZUrWEl5SlLdXHK1Z6q9c5+v660qUYXycw64zVEeiBp9+1aSul9ebwm6SsNKI7yEDGR8c1fSWw6okHBPSs+A45+H1D4qiQiM8bXAJPPl9iuZdoGUzjkTaqhga6dpFzfSS0A+74G+wFyCOhtdeNNqWFKSpI5fPxrwPLH4qD70CnIgoKBlwhXjjpRTBUN0OA+8Yq0ftFg88jhNKCL2F2Hb/qseeb7Ll/7EZnpII3UsDmm13ESN3vuPduCCBsRvdJGRzDCmWsexOKAcY8Y4/4jSnZSm+6sH4/314pUpJ9YA/AVjdWUbxamcy3gJ3sP0AWwzDcUiOquZLq8TRxygD/mLj9ls22dwVMN9Kh7Z3BUw30rlC9EI9ChQoQmz/drP3pZX+z6c0JAn3ye1EjqvDTaVrzussvEAAbk4B29laBf7tZk9NeDpaXofTcvWsyOzZLZqdidLaeKx8qSiHLCWU8mVkqUpOyRnAVWzRzmmnbK3kFRGO4TFjuHS4dMSGyCxI5Xzl4vazuI1rNuukdSXGCxNab5lxZDjBd5QQOblIJxvjPnVz8CZ2oJVzUxdb5cJ7SLTHdUJUlbg51HrhRO586zFxFlsR9UPQLfIRMixlKZjvpSQH2kuKCHAFDIyMHB33rUvo8ocdkz0Os5cj2+I1kb+qAo71W884jVxsvBI4E77Ej7LoHZ1l/DI6cU8kLXhjQ27mgk223uPBXraYseXIQhTSVAqwciojjAiRpvR92uVgWiHOjMdo04pIUEnmHgoEHb2V02mmczW0BO5UNiK4/0opPyXh5diFhtbqmow3wPWWkVV8Ex3F2wt01Uly8D43fqrVimU8uSvIloITt1iYf/ABVGcS+LHELRsfnjvoadRAivuJfiNkdo44EnGD0wD5VWx9LLiewSFN2V3A25oit/oWKkOOt7RLbcZ+XRpfJGgx+ZlznIKQSQog4JyKoWQMr+FdwfilfGABM7jxK5DWZNy85x/gox6MA+1lvvg3q+9684e27Vt/TETJnrfwiK0pCEJQ6psD1lKJPqE5z49Nq5y+cd7YjXCNHacjuT0REuruEphAcCShJ/BtjIBwe8roMYAJ3GeeG/FPiGLPb+F1imNtW9yQ5yuoZw8htfMtTfadAgqKjnHNuRnG1F4UaeukfiNcrbcHEvNw4Upx9TLmUkcowOYdRkip+TMEstNHDBfVYanHm9t7fPqqJgXZbTMxaevxJrXRa3d3GPh0321bdBYadx43UXxYlO3fURuJecd7VtKwtxfMpQOVZJ+NdvpONHTDhRGnRLfERkJLSCpPR7GPpUP/7TC63PR7OnLdq+PZBMmzXDHDE0YZbS2nHqoSRzDPmfPandx4lR7ciI9AgsMjsmlLZZb5EhZC+ZIHQAHk+k1qYPUNopJZJDyOq9EZVrafCHyveABpsP7AKftcS3XHV82I7KCl2uHznKeYNulYQUn28pI9mSRvVea+kXG/39DMKI+7Ejp5IwQ2SHMn1nBjYgnGD5BPnUUzrSXFuN9ntsJC73zgEq/wA1zLySPM4yB5Hek4uqWgWWYhDRbCDlOVLDiUJRzA7AeqkDGDUZNUsmBMzrKhRULK/G58VxF54AYBv6psu3TIaVrkoDakPBlTWfXSSnmyfDHxz7K9EiLFOZEhKCRnGd6s3hvwjuvEiSu4TnnoVrTguyHfWedPQBAOw2GM+FWjdfR54foty47NqIcSnIfK1FzI8ebP8A+qgp8VggdaPcKVdAyV/7nYeazpp2JqHV01MDSNoeeUTyqfUnCE+Z32q0NJejauXdg/r++KktoUFJixQUJV71f3V0fDt226IccsrzIbIcUE+r1OdjXewb6mbeGY8dJXkkqI6AVFvxWaofYbBPlpGsjLr7rubHbrfZbZHtdqitx4sZsNNNIGEpSPCpGOr1z5eFNIwKUJHXIpzH2cV7aVziTdRLR4p+FJopUR0O1EKsdKKpflSjhBIHKOpeRkmklqz0oqlE0QqKTiiyNTVtC2dwVMN9Kh7Z3BUw30pVgR6FChQhNn+7WNPuls9i3cGdPvyFcqTq2KjOM4JhzP7q2W/3axh90ztTV44L6ehvOltH6r4iiQM9Icz++ssN+8Fk13C+bOqHG3b8S2kKSlDQG3+iCfz1sT0cHy2L9KYY50qZhsk9OUpbOfz1ju9w1y9TvR2SCUlsg9AAEpH9lbP9F8MSrDqGVzp51SWmikHolLYx+c1AZrBJLh0BXQskEMa5pV16ekhN2adxhOc4qr/Sou4GiXkKQhaH7gyjChnIK8/2VYkOQmC/2qkhXsqlvSeu0a36VgTpyA4z85tLU2MZUBk4APXpVBwGqifLDCDvrV0rYdQdKfBUZxB0zP1BfZdgZuFtjvPSo4CVENoQOQ9E7k9a59nh3pSyQ79Invv3mbYwgOoCuyjFRUAU+qeYj4jpUHrDiJKu+vXNXQYio2HG1NNPJ3CUAAZHTfH11L2S5aEmWi56g1rqGeq5zXHFptkNBQ26rqkrI2x7CdsV35sjL3XLJQ18lyp2RxBgackWNNrixbdHVDU7KjwmgUc6goJCs7k4wdzVeadmahFxukuzXF2IH2HlSnUJGOzVuQfLNclcLrGbcWplZKVqPKgK5iBnYfRS1riapuuUQGVx47mMqUeXI88eNYH1rWlaxeGmwUjaL6qwTY059tmYmGvtER5OVtKI800vq/irc9aXBEyfDjtJb/Ashpvs2Wk9eUY+nBqQtnDeESl66ylSlHqN0jNDibEiwdKW23Q47bTaJoV6oxn8Gsf21gp6s1FQ2K2xKdRWqKhsTuCVGWnTNw1QBIjKUtAPKVIPI2D7/GrQ05w7tkWzSpQgRxPhoS4lxorVknqDzE/2VzHCMlOmHMfvpz8yas+yXFMaJOaKcqeShI+k10HHcvUUWWX1rG/vLCx8N12qsyphdPlg4gyP95YG/huu50DxEtsC0tWwNL7VpOFBKMb9TXWS9ZQnonaLXyDxB61zOk9MQkwEy5cZCnHvWwR0HhvUy/oyyzcF9tfKFZ5UrIB9lcW9lc4A3XF/bYmkhG0xa7Ze4z8mTDbdbU4Q2Vp6jxx8anoVlt9rdPyOOhor68vjQiNsREJYYbDaEDlSE7AClXH/AMInettkTWCyj5qlz3bcKbiOEISM5p4ys83X2VFxXMBO9PGXT2h26CstliJvwn3OfdRCrAyaTLm2dqIXfOhYybo6ncdKSU4d/WpCTISykqJFQNxvWUqS24AaELfts7gqYb6VD2zuCphvpQmI9ChQoQmz/drHv3R3I4QWBQ/F1XFJ93ySXWwn+7WN/ulClo4MWMo6/qrjf1OXWSJ2l4Ka4XFl824akzNbyOYgJUvlwOngK2H6O8J/TFjvcJ1sEvTErbUMABHJvWG494VZ9VrkPHAUokZ8D1BrTlq48aLg6Zjyn7t2cgoT2zbaRzlXTu56VBZh7yf4G3BV4yrUR04JcbELT8NyM+4TIXtgkBJqgvS6mW6PpO2ImlTkf5eFq5COYgIVtvVb3v0ptQTkrt+grFIDhGBMfUAlPtA6H4mq2vY1nrmSmVrzUsiajPMGUq9VJP0AfAVVMIwGohqWSloABv5qy4tmKlZA6NhuSFwt21J87z3JSW3XCs4SAO6kdB8BtSsPSl3uywqQoxWwfPKiPdXcwrJbICS1DjBA9u5+mnyGw2N9zXSnTucuZyS6jcKHs+i7Fb+VbkYPvgg9o5vg10LcdAxygAeyiAKJChj2U4bBArB7xO61zJdKNkNjH11yPFRwqs0EeAlD4+oquuTgnBrkeKqP+x4OP30P6CqkcMsKtnqt7CD/ABsfqpLhOf8AJpwD99L/AKKatHSVrF0ugbXns2ilasfVmqs4UHl044PKUv8AopqxbNrE6TmocTZnbgmUeRQbcCS2BvncYPXxIrueJ0slblXuYhdxaLD5r0tiNNNV5M7mnbd5aLAequ6EkIQlIwkAdPCpFJGOtRNomC5W9ieiO4yH0BYQ5jmAPTOCRUm1k7HFefnsdG4scLELy1LG+J5Y8WINinLaUnBosgBK0kbV6ghO1JTFBJSSaYTZMUpCV3c7bfXUjHWVLVzDc1EQVbJzuTvT1MkNqV5AUqcNipBTiQOuPjTKRPabBKtgKYT7shA2IGOu9c3OuTi8kLODQlcBZSN1vCV+q2s4xvtXPPyi5tnek1vqWSVE70gpe9JdNtdfTS2dwVMN9Kh7Z3BUw30pVjR6FChQhNn+7WQPujTaXeDlmSsZH6qIx/8AxJda/f7tZC+6L/sO2b/zPH/qkqhHVfNW5aTtd1c7d/nDoGAU7UjH0PamSFuIW4oYPrqyPoqfzynB60ZKt8nNFrrMyQs4Qjttxmw000lKR4AYpVR5h0ovXoKMlSQkgjOelN0pHOLjdNFKHOSD40qhQVSLnKFnp9Fet4yTnrTrJtyE4SSCN9qcJUQcHemicEghRpyketkUJAbpfmKEFxLfMoDYeZrgZV8RraejTz7HYOodPK256pSsZByfHxrv2z4EdarriNZnbLd4uqraCjtFp7UpzhLo6H4gfSPbU9l6aCKraJm3VkyxUU8Fa0VDQR/7srB0vp5GnLaqCl4O87hcKsYwSAMD6K6O1QV3K4xoCB/nnUoPsT4n6KhNPXdq/wBpj3NkjDqfXH7lY6irC4b2pcy6OS0FKS0EtNqUcAOLOAc+wZrv+J19PhOCPqIPhDfd9TwvUeL4lS4Jl19XTbMDPd9Tx+as2M5HispaKkpShISBnpjoKlI8eW7FVcExiIyMZcUtKSQSkAhJIUoesncAgcwyRkVKNy9F6ec+UWtHzm+xcFvs9mr1+yMZKQguLb2SHFOHxzyj1dwoRLlzU5DTbo0RtiK0hLLJUSt5LSQj1CvYEFTaVd3OfHFeY3OLyXHkrx297pHF7uSlG+VRyKSngZSfIE16h0ik5TmUn2A0wlHw8paDKDaAeYZAxvTFd8SouJC8EHFQr9yWgEIWR4GmDMjmWrPjvS8BIfeKkpE110klW2abFZPeVSZcBHWkVujHXpQeE7SlS6kHFJOOZOxpFTvU0RThpqcvqJbO4KmG+lQ9s7gqYb6U9a6PQoUKEJs/3ayF90W/Yes3/meP/VJVa9f7tZD+6LfsO2XP/wAzx/6pKoR1Xzv9u1eZPgM16ceFeAY8aE9egkV4tZAwBXuR1NFUpOMUqE2J9Y5FHByNqJ1UcV6CEnB60iE4aCkp9al21KyBvSA3G1Lt9QDQhOm/DepmJo+Nq62zLZOThh9soCvFK/xVD2g4NRkKK7JcDbKSpR6CrPsFvTCittDGcZV76c17o3BzeQlY8xuD28hZx0FcJWktTzdFXkhBDymd9gl1J8PYoY+qtS6Ht3yKwtFxGFSSXlA+R6fUBTO56T0vfild5sUOW4ndLq2h2ifcsesPga6SKtCG0tIGEpSEpHkB0q0YhmaWuwxuHEbA3/wrliWcJ8QwVmEkWAdf5eH1TxJAGwA91Om1nG9MgelKoUoeNVVUxPeY5zSUl3swSRnY5FF7TYE9RTC6yezZUsnqDSJCubMhTjih0HMcivWlkOqA8ajmHlBRBPU5py2rKs53pU9qf83iaRUrJJztRStWOo+mkl77+NCejLWNsUTn8zRSD50mtQB5c00osvqlbO4KmG+lQ9t7gqYb6U5a6PQoUKEJB5OU1mz01+F2ueKvDO12Dh/Y/nS4R78xNdZ+UsscrKY8hBVzOrSk+s4gYBzv02NaXWnNM34/NnIoQvlAr0OPSQT3uHB/neB9vSZ9Dv0kM7cOT/O0H7avqq5bkqPSifNafZQi5XyrV6HXpIn/AOHB/neB9tRT6HPpI4/Y2Of9bwPtq+qvzWmh81poRqK+U3/Q39JMKJ+9sf53gfb0f/ob+kiSD97c/wA7wPt6+q3zWmh81poRqK+V6PQ79I4AZ4ckf72g/bUsn0PvSMB/Y7OP9bQftq+pXzWmh81poRqK+aenvRL47xZAen6DKAkbD5zhnf4O12UX0cuMzXe0YR/vCL9pW+vmtND5rTQi6wo16PnF9OObR5/5+N9pTlPALi4Mf5Ikf/fRvtK3F81pofNaaEXKxGngPxaxhWkj/wA9G+0o6eBHFgH9ah/52N9pW2fmtND5rTQjUVikcC+K4/8AdM/87G+0qJvfo/8AGSSlIi6PLnmPl8UY+lyt2fNaaHzWmhGor57tejVxuSok6II6/tjE+1pdn0buNST62iyB/rCJ9rX0B+a00PmtNCNRWBT6OXGcj9Zh/nCL9rSZ9HDjT4aLP84Rfta3981pofNaaE7vCvn+r0b+NZ7uiyf94RPtaKPRk45On1NDn+cof2tfQMWxI8KcMwUoOyaSyUSFGgt8qQMVJoG1IstctOAMClTF7QoUKEIUQgdKFChCKUJ64ovKnyoUKEIcqfKhyp8qFChCHKnyocqfKhQoQhyp8qHKnyoUKEIcqfKhyp8qFChCHKnyocqfKhQoQhyp8qHKnyoUKEIcqfKhyp8qFChCHKnyocqfKhQoQhyp8qHKnyoUKEIcqfKjhCR4UKFCEZIFGoUKEIUKFChC/9k="
LOGRO = "..."
html = f'''<div style="display:flex; gap:28px; align-items:flex-start; flex-wrap:wrap; font-family:sans-serif;">
  <div style="text-align:center;">
    <img src="data:image/jpeg;base64,{PROTO}" style="height:260px; border-radius:10px;">
    <div style="margin-top:8px;"><b>💭 Prototipo</b><br>lo que queríamos lograr</div>
  </div>
  <div style="text-align:center;">
    <img src="data:image/jpeg;base64,{LOGRO}" style="width:640px; max-width:100%; border-radius:10px;">
    <div style="margin-top:8px;"><b>✅ Resultado</b><br>lo que logramos</div>
  </div>
</div>'''
display(HTML(html))

### 🔎 Cómo leer la captura

| Ventana | Qué es |
|---|---|
| 🖥️ Terminal de abajo | **`main.py`**: recibe los datos del celular (Phyphox) y los guarda en MongoDB. Cada línea `[giro]` es un lote de ~50 muestras. |
| 🖥️ Terminal de arriba | **`mando_local_todo.py`**: lee los datos nuevos, calcula el ángulo y **decide el comando** (→ ARRIBA, → ABAJO…). |
| 🎮 Derecha | El juego (Temple Run 2) recibiendo las flechas como si vinieran del teclado. |

> 📌 En este cuaderno **no se pulsa ninguna tecla**, pero los datos son **reales y en vivo**: Colab lee lo que llega a MongoDB, hace todo el procesamiento y muestra qué comando detecta. Las teclas las pulsa el programa que corre en mi computador.

## 2. 📡 ¿Cómo llegan los datos?

In [ ]:
#@title 🗺️ Recorrido de los datos { display-mode: "form" }
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 3.4))
ax.axis("off"); ax.set_xlim(0, 13); ax.set_ylim(0.1, 3.3)
cajas = [
    ("Celular\nPhyphox",  "mide ω\n(giroscopio)",   AZUL),
    ("main.py",           "recibe los POST",        AZUL),
    ("MongoDB\nAtlas",    "colección\ngiroscopio",  VERDE),
    ("mando_local\n_todo.py", "ω → θ → comando",    NARANJA),
    ("Juego",             "flechas ↑ ↓ ← →",        ROJO),
]
x, ancho, sep = 0.2, 1.6, 1.15
xs = []
for nombre, sub, col in cajas:
    ax.add_patch(FancyBboxPatch((x, 1.4), ancho, 1.4, boxstyle="round,pad=0.04,rounding_size=0.15",
                                fc=col, ec="none", alpha=0.92))
    ax.text(x + ancho/2, 2.1, nombre, ha="center", va="center", color="white", fontsize=12, fontweight="bold")
    ax.text(x + ancho/2, 1.25, sub, ha="center", va="top", fontsize=9.5)
    xs.append(x); x += ancho + sep
etiquetas = ["POST\ncada 0.25 s", "guarda\nel lote", "consulta\ncada 0.05 s", "pynput\n(teclado)"]
for i, e in enumerate(etiquetas):
    x0, x1 = xs[i] + ancho + 0.06, xs[i+1] - 0.06
    ax.add_patch(FancyArrowPatch((x0, 2.1), (x1, 2.1), arrowstyle="-|>", mutation_scale=18, lw=2, color=GRIS))
    ax.text((x0 + x1)/2, 2.35, e, ha="center", va="bottom", fontsize=8, color=GRIS)
# marco: lo que explica este cuaderno
ax.add_patch(FancyBboxPatch((xs[3] - 0.08, 0.55), ancho + 0.16, 2.4, boxstyle="round,pad=0.03,rounding_size=0.2",
                            fc="none", ec=NARANJA, lw=1.6, ls="--"))
ax.text(xs[3] + ancho/2, 0.25, "↑ esto es lo que explicamos y simulamos aquí", ha="center", va="center",
        fontsize=9.5, color=NARANJA, fontweight="bold")
plt.show()

**Paso a paso**

1. **Phyphox** (en el celular) mide la velocidad angular ω en cada eje y la manda con un `POST`.
2. **`main.py`** (en mi PC) recibe cada paquete y lo guarda en **MongoDB Atlas**.
3. **`mando_local_todo.py`** consulta Mongo cada 0.05 s, pide solo los documentos nuevos (`_id` mayor al último visto), calcula el ángulo y decide el comando.
4. **`pynput`** pulsa la tecla en el juego.

Cada documento es un **lote** de ~50 muestras que llega cada ~0.25 s. Dentro del lote, las muestras están separadas unos **4.8 ms (≈ 208 muestras/s)**. Ese tiempo entre muestras es la **h** de nuestro método numérico.

In [ ]:
#@title 🔌 Conexión a MongoDB (los mismos datos que usa el mando) { display-mode: "form" }
import subprocess, sys
try:
    import pymongo
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo"], check=True)
    import pymongo
from pymongo import MongoClient

try:                                                   # 🔑 mejor guardar la URI en los Secretos de Colab (candado ⟵)
    from google.colab import userdata
    URI = userdata.get("MONGODB_URI")
except Exception:
    from getpass import getpass
    URI = getpass("Pega tu MONGODB_URI (no se muestra en pantalla): ")

col = None
try:
    cliente = MongoClient(URI, serverSelectionTimeoutMS=8000)
    cliente.admin.command("ping")
    col = cliente["trayectoria_numerica"]["giroscopio"]
    print(f"✅ Conectado a MongoDB · {col.estimated_document_count()} lotes en la colección")
except Exception as e:
    print("❌ No se pudo conectar:", type(e).__name__)
    print("   → En MongoDB Atlas: Network Access → Add IP Address → 'Allow access from anywhere' (0.0.0.0/0).")
    print("   → Revisa también que la URI sea correcta.")

In [ ]:
# El último lote que llegó a Mongo (datos reales)
doc = None if col is None else col.find_one(sort=[("_id", -1)])
if doc is None:
    print("⚪ Todavía no hay datos: revisa la conexión y dale play en Phyphox.")
else:
    print(" Un lote (documento de Mongo):")
    print("   timestamp :", np.round(doc["timestamp"][:4], 3), "…")
    print("   gy        :", np.round(doc["gy"][:4], 4), "…")
    print("   gx        :", np.round(doc["gx"][:4], 4), "…")
    dt_medio = np.mean(np.diff(doc["timestamp"]))
    print(f"\n   muestras por lote : {len(doc['timestamp'])}")
    print(f"   duración del lote : {doc['timestamp'][-1] - doc['timestamp'][0]:.2f} s")
    print(f"   h entre muestras  : {dt_medio*1000:.1f} ms  →  {1/dt_medio:.0f} muestras/s")

> ℹ️ Además de `timestamp`, `gx` y `gy`, `main.py` guarda `recibido_en` (hora de llegada). El mando la usa para mostrar `lote llegó hace X s` y vigilar la **latencia**.

## 3. 🧭 El modelo: una EDO con valor inicial

El giroscopio entrega **ω(t)** (qué tan rápido gira). Nosotros queremos **θ(t)** (cuánto giró). Es como tener el velocímetro de un carro y querer saber cuánta distancia recorrió.

$$\frac{d\theta}{dt} = \omega(t), \qquad \theta(0) = 0$$

- **θ(t)** → la incógnita: el ángulo.
- **ω(t)** → el dato: lo mide el sensor.
- **θ(0) = 0** → condición inicial: el ángulo con el que empieza el celular lo definimos como cero. Por eso es un **problema de valor inicial (PVI)**.
- Es una **EDO de primer orden**.

Su solución exacta sería $\theta(t)=\int_0^t \omega(s)\,ds$, pero ω **no es una fórmula**: es una lista de números medidos cada ~4.8 ms. No se puede integrar a mano, así que necesitamos un **método numérico**.

> 💡 Casi cualquier fenómeno del mundo real se puede plantear como una EDO. Esta es de las más simples: ideal para probar los métodos que vimos en clase.

## 4. 🧮 Métodos numéricos: de Euler a Heun

Avanzamos de una muestra a la siguiente, separadas por un paso $h$ (en el código, `dt`).

### Euler
Usa solo la pendiente del **inicio** del intervalo:

$$\theta_{n+1} = \theta_n + h\,\omega_n$$

### Heun (predictor–corrector, RK2) — lo que vimos en clase
Promedia la pendiente del inicio y la del final:

$$K_1 = f(t_n,\theta_n) \qquad K_2 = f(t_n+h,\ \theta_n + hK_1)$$

$$\varphi = \frac{K_1+K_2}{2} \qquad \theta_{n+1} = \theta_n + \varphi\,h$$

### ¿Qué pasa en nuestro problema?
Aquí $f(t,\theta)=\omega(t)$ **no depende de θ**, así que

$$K_1 = \omega_n \qquad K_2 = \omega_{n+1}$$

El predictor ($\theta_n + hK_1$) no cambia nada y Heun se reduce a la **regla del trapecio**:

$$\boxed{\theta_{n+1} = \theta_n + \frac{\omega_n + \omega_{n+1}}{2}\,h}$$

| Método | Pendiente que usa | Error global |
|---|---|---|
| Euler | ωₙ (inicio) | O(h) |
| Heun / trapecio | (ωₙ + ωₙ₊₁)/2 | O(h²) |

**¿Por qué no RK4?** Necesita ω en el punto *medio* del intervalo, y el sensor solo entrega muestras discretas.

In [ ]:
def euler(f, t, th0=0.0):
    th = np.empty_like(t); th[0] = th0
    for n in range(len(t) - 1):
        h = t[n+1] - t[n]
        th[n+1] = th[n] + h * f(t[n], th[n])
    return th

def heun(f, t, th0=0.0):
    th = np.empty_like(t); th[0] = th0
    for n in range(len(t) - 1):
        h = t[n+1] - t[n]
        k1 = f(t[n], th[n])                       # pendiente al inicio
        k2 = f(t[n+1], th[n] + h * k1)            # pendiente al final (con predictor)
        th[n+1] = th[n] + h * (k1 + k2) / 2       # φ = (K1 + K2) / 2
    return th

def trapecio(acum, f_ant, f_nueva, dt):           # ← la versión que usa el mando
    return acum + 0.5 * (f_ant + f_nueva) * dt

# ¿Heun completo = trapecio cuando f solo depende de t?
t = np.linspace(0, 5, 51)
th_heun = heun(lambda t, th: np.cos(t), t)
th_trap = np.zeros_like(t)
for n in range(len(t) - 1):
    th_trap[n+1] = trapecio(th_trap[n], np.cos(t[n]), np.cos(t[n+1]), t[n+1] - t[n])
print(f"Máxima diferencia entre Heun y trapecio: {np.abs(th_heun - th_trap).max():.1e}")

## 5. 🧹 Del sensor real al ángulo: calibración, zona muerta y re-centrado

Un sensor real no mide solo ω:

$$\omega_{medida} = \omega_{real} + \text{sesgo} + \text{ruido}$$

Si integramos eso tal cual, el sesgo y el ruido **se acumulan** y el ángulo "se va solo" aunque el celular esté quieto (**deriva**). Por eso, antes del trapecio, cada muestra pasa por estas etapas:

| # | Etapa | Qué hace | Parámetro |
|---|---|---|---|
| 1 | **Calibración** | Con el celular quieto, la media es el sesgo y la desviación estándar es el ruido. Si hay demasiado ruido, recalibra. | `T_CAL = 2 s` · `CAL_MAX_STD = 0.10` |
| 2 | **Quitar el sesgo** | Resta la media a cada medición. | |
| 3 | **Zona muerta** | Si ω es menor que el umbral, vale 0: el ruido no se integra. | umbral = máx(0.05, 3σ) |
| 4 | **Integración** | θ ← θ + ½(ωₙ + ωₙ₊₁)·h  (trapecio) | |
| 5 | **Re-centrado** | Si el celular lleva 0.6 s quieto, θ vuelve a 0. | `T_REPOSO = 0.6 s` |

In [ ]:
# ---------------- Parámetros (los mismos de mando_local_todo.py) ----------------
CAMPOS = ("gy", "gx")              # gy: ARRIBA/ABAJO  |  gx: IZQUIERDA/DERECHA
SIGNO  = np.array([1.0, -1.0])     # -1 en el eje que salga invertido
T_CAL, CAL_MAX_STD, UMBRAL_W = 2.0, 0.10, 0.05
T_REPOSO = 0.6
# T_CAL: segundos quieto para calibrar  |  CAL_MAX_STD: ruido máximo permitido al calibrar
# UMBRAL_W: zona muerta mínima (rad/s)  |  T_REPOSO: segundos quieto para volver a θ = 0

def nuevo_integrador():
    """Estado inicial del integrador (un diccionario)."""
    return dict(t=None, t0=None, w=np.zeros(2), th=np.zeros(2), buf=[],
                sesgo=np.zeros(2), umbral=np.full(2, UMBRAL_W),
                calibrado=False, t_act=None, estado="calibrando")

def paso(g, t, w_cruda):
    """Procesa UNA muestra: calibra, limpia ω e integra con el trapecio."""
    w_cruda = np.asarray(w_cruda, dtype=float) * SIGNO    # corrige la orientación de los ejes
    if g["t0"] is None:                                   # primera muestra
        g["t0"] = g["t"] = g["t_act"] = t
        g["buf"] = [w_cruda]
        return True
    dt = t - g["t"]
    if dt <= 0:                                           # intervalo inválido: se ignora la muestra
        return False

    if not g["calibrado"]:                                # 1) calibración: el sesgo es el promedio con el celular quieto
        g["buf"].append(w_cruda)
        g["t"] = t
        if t - g["t0"] >= T_CAL:
            datos = np.array(g["buf"])
            if datos.std(axis=0).max() > CAL_MAX_STD:     #    demasiado ruido: se movió
                g["estado"], g["t0"], g["buf"] = "recalibrando: deja el celular quieto", t, []
            else:
                g["sesgo"] = datos.mean(axis=0)           # promedio a lo largo de las filas
                g["umbral"] = np.maximum(UMBRAL_W, 3.0 * datos.std(axis=0))
                g["calibrado"], g["t_act"], g["estado"] = True, t, "listo"
        return True

    w = w_cruda - g["sesgo"]                              # 2) quitar sesgo
    w = np.where(np.abs(w) < g["umbral"], 0.0, w)         # 3) zona muerta

    g["th"] = trapecio(g["th"], g["w"], w, dt)            # 4) UNA integración: ω → θ
    g["w"], g["t"] = w, t

    if np.any(w != 0.0):                                  # 5) re-centrado en reposo
        g["t_act"], g["estado"] = t, "en movimiento"
    elif t - g["t_act"] >= T_REPOSO:
        g["th"] = np.zeros(2)
        g["estado"] = "en reposo"
    return True

def grados(g):
    return np.degrees(g["th"])

In [ ]:
#@title 🔍 Celular quieto: sin calibrar vs con calibración { display-mode: "form" }
# Simulación de un celular QUIETO: ω_medida = sesgo + ruido  (ω_real = 0)
rng = np.random.default_rng(0)
h = 0.0048                                  # s entre muestras (≈ 208 muestras/s)
t = np.arange(0, 12, h)
sesgo_sim, ruido_sim = 0.05, 0.02           # rad/s (valores ilustrativos)
w = sesgo_sim + ruido_sim * rng.standard_normal((len(t), 2))

# a) Sin calibrar: integrar la medición tal cual (eje 0)
th_sin = np.zeros(len(t))
for n in range(len(t) - 1):
    th_sin[n+1] = trapecio(th_sin[n], w[n, 0], w[n+1, 0], h)
th_sin = np.degrees(th_sin)

# b) Con calibración, zona muerta y re-centrado (la función paso)
g = nuevo_integrador()
t_con, th_con = [], []
for n in range(len(t)):
    paso(g, t[n], w[n])
    if g["calibrado"]:
        t_con.append(t[n]); th_con.append(grados(g)[0])

plt.figure(figsize=(9, 4))
plt.plot(t, th_sin, color=ROJO, lw=2, label="Sin calibrar")
plt.plot(t_con, th_con, color=VERDE, lw=2, label="Con calibración + zona muerta")
plt.axhline(25, color=GRIS, ls="--"); plt.text(0.2, 26.5, "umbral de disparo = 25°", color=GRIS)
plt.xlabel("t (s)"); plt.ylabel("θ (grados)")
plt.title("Celular quieto: ¿se va el ángulo?"); plt.legend(loc="upper left"); plt.show()

if (th_sin >= 25).any():
    print(f"Sin calibrar: cruza 25° a los {t[np.argmax(th_sin >= 25)]:.1f} s (comando fantasma).")
print(f"Con calibración: el ángulo máximo es {np.abs(th_con).max():.2f}°.")

🔍 **Simulación de un celular quieto** (ω real = 0; solo sesgo de 0.05 rad/s y ruido). Sin calibrar, el ángulo **cruza el umbral de 25° con el celular quieto**: el mando dispararía un comando fantasma. Con la calibración y la zona muerta, el ángulo se mantiene en 0.

## 6. 🚦 Del ángulo al comando: histéresis

Ya tenemos θ en grados para dos ejes: **↕ pitch (gy)** y **↔ roll (gx)**. Falta decidir cuándo eso es un comando.

Si usáramos un solo umbral (25°), un ángulo que oscila cerca de 25° dispararía la tecla decenas de veces. Por eso usamos **histéresis** (disparador de Schmitt):

- 🔴 **Dispara** cuando el ángulo máximo supera **TH_ON = 25°**.
- 🔒 Queda **bloqueado** (no vuelve a disparar) hasta que baje de **TH_OFF = 10°**.
- ⏱️ **COOLDOWN = 0.35 s** mínimo entre comandos.
- 🔄 **T_REARME = 0.25 s**: si la mano queda quieta tras el gesto, se re-centra θ y se rearma.

El eje con mayor ángulo decide la dirección:

| Eje dominante | Signo del ángulo | Comando |
|---|---|---|
| pitch (gy) | positivo | ⬆️ ARRIBA |
| pitch (gy) | negativo | ⬇️ ABAJO |
| roll (gx) | positivo | ➡️ DERECHA |
| roll (gx) | negativo | ⬅️ IZQUIERDA |

> En la demo real, cada comando se traduce en la tecla de flecha correspondiente.

In [ ]:
TH_ON, TH_OFF = 25.0, 10.0         # grados: disparo / rearme
COOLDOWN      = 0.35               # s mínimos entre comandos
T_REARME      = 0.25               # s quieto tras un gesto => θ vuelve a 0

class Comandos:
    """Histéresis (disparador de Schmitt): dispara al cruzar TH_ON, rearma bajo TH_OFF."""
    def __init__(self):
        self.activo, self.t_ult = None, 0.0

    def evaluar(self, ang, ahora):
        pitch, roll = ang
        mayor = max(abs(pitch), abs(roll))
        if self.activo is None:
            if mayor >= TH_ON and ahora - self.t_ult >= COOLDOWN:
                if abs(pitch) >= abs(roll):                       # eje dominante
                    cmd = "ARRIBA" if pitch > 0 else "ABAJO"
                else:
                    cmd = "DERECHA" if roll > 0 else "IZQUIERDA"
                self.activo, self.t_ult = cmd, ahora
                return cmd
        elif mayor < TH_OFF:                                      # rearme
            self.activo = None
        return None

## 7. ▶️ Datos reales en vivo: ¡aquí salen los comandos!

La celda de abajo se conecta a **mi MongoDB**, lee cada lote en cuanto `main.py` lo guarda, lo integra con **el mismo código** del mando y **dibuja en vivo** el ángulo θ(t) de *pitch* y *roll*, los umbrales ±25° / ±10° y, cuando se cruza el umbral, una **flecha grande** con el comando.

**Cómo usarla**
1. Ejecutar *en orden* todas las celdas anteriores (Runtime ▸ Run before) para que existan la conexión `col`, `nuevo_integrador`, `paso` y `grados`.
2. `main.py` corriendo en mi PC (recibiendo del celular).
3. Ejecutar la celda de abajo, dar *play* en Phyphox y **no mover el celular ~2 s**. Esperar `Estado: listo`.
4. Mover el celular: ⬆️ ⬇️ ⬅️ ➡️ y ver cómo sube el ángulo hasta disparar el comando.

> ⏹️ Para detenerla antes, botón **Stop** de la celda. Se detiene sola a los `DURACION` segundos.

In [ ]:
#@title ▶️ En vivo: ángulo y comandos mientras mueves el celular { display-mode: "form" }
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

if "col" not in globals() or col is None:
    raise RuntimeError("Primero ejecuta la celda de conexión a MongoDB (sección 2).")

requisitos = ("trapecio", "nuevo_integrador", "paso", "grados", "CAMPOS", "Comandos",
              "TH_ON", "TH_OFF", "COOLDOWN", "T_REARME", "AZUL", "NARANJA", "ROJO", "GRIS")
faltantes = [nombre for nombre in requisitos if nombre not in globals()]
if faltantes:
    raise RuntimeError(
        "Faltan definiciones: " + ", ".join(faltantes) +
        ". Ejecuta antes: Preparación, conexión (sección 2), métodos (sección 4), sección 5 y sección 6."
    )

DURACION = 180      # s que escucha la celda
VENTANA  = 8.0      # s que se ven en la gráfica
REFRESCO = 0.3      # s entre redibujos
SONDEO   = 0.05     # s entre consultas a Mongo
FLECHA = {"ARRIBA": "↑", "ABAJO": "↓", "IZQUIERDA": "←", "DERECHA": "→"}

def procesar_doc(doc):
    global g
    cmds = []
    listas = [doc.get(c, []) for c in CAMPOS]
    for t, *w in zip(doc.get("timestamp", []), *listas):
        if g["t"] is not None and t < g["t"] - 1.0 and t < 1.0:
            g, clf.activo = nuevo_integrador(), None
        if paso(g, t, w) and g["calibrado"]:
            hist.append((t, *grados(g)))
            cmd = clf.evaluar(grados(g), t)
            if cmd:
                cmds.append((t, cmd))
            elif T_REARME and clf.activo and t - clf.t_ult >= T_REARME and not np.any(g["w"]):
                g["th"] = np.zeros(2); clf.activo = None
    return cmds

def dibujar(ultimo_cmd):
    """Solo actualiza los datos de la figura (no la vuelve a crear): así no titila."""
    if hist:
        h = np.array(hist[-int(VENTANA * 220):])
        x = h[:, 0] - h[-1, 0]
        l_pitch.set_data(x, h[:, 1])
        l_roll.set_data(x, h[:, 2])
        for m in marcas:
            m.remove()
        marcas.clear()
        for t, c in eventos:
            if t >= h[0, 0]:
                marcas.append(ax.axvline(t - h[-1, 0], color=ROJO, alpha=0.5))
                marcas.append(ax.text(t - h[-1, 0] + 0.05, 85, c, rotation=90, va="top", color=ROJO, fontweight="bold"))
    ax.set_title(f"Estado: {g['estado']}")
    cmd, tc = ultimo_cmd
    if cmd and time.monotonic() - tc < 1.2:
        t_flecha.set_text(FLECHA[cmd]); t_flecha.set_color(ROJO); t_cmd.set_text(cmd)
    else:
        t_flecha.set_text("·"); t_flecha.set_color(GRIS); t_cmd.set_text("")
    pantalla.update(fig)

g, clf, hist, eventos = nuevo_integrador(), Comandos(), [], []

# Figura: se crea UNA sola vez
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [3, 1]})
plt.close(fig)                                   # que Colab no la muestre sola; la mostramos con display
l_pitch, = ax.plot([], [], color=AZUL, lw=2, label="↕ pitch (gy)")
l_roll,  = ax.plot([], [], color=NARANJA, lw=2, label="↔ roll (gx)")
for y, ls in ((TH_ON, "--"), (-TH_ON, "--"), (TH_OFF, ":"), (-TH_OFF, ":")):
    ax.axhline(y, color=GRIS, ls=ls, lw=1)
ax.set(xlim=(-VENTANA, 0), ylim=(-90, 90), xlabel="t (s, 0 = ahora)", ylabel="θ (grados)")
ax.legend(loc="lower left")
ax2.axis("off")
t_flecha = ax2.text(0.5, 0.55, "·", ha="center", va="center", fontsize=90, color=GRIS, fontweight="bold")
t_cmd    = ax2.text(0.5, 0.1, "", ha="center", fontsize=16, color=ROJO, fontweight="bold")
marcas = []                                      # líneas rojas de los comandos
fig.tight_layout()
pantalla = display(fig, display_id=True)         # salida que se va actualizando

ult = col.find_one(sort=[("_id", -1)])
ultimo_id = ult["_id"] if ult else None
ultimo_cmd, t_dib = (None, 0.0), 0.0
print("🟢 Dale play en Phyphox y deja el celular quieto ~2 s (calibración)…")
fin = time.monotonic() + DURACION
try:
    while time.monotonic() < fin:
        filtro = {"_id": {"$gt": ultimo_id}} if ultimo_id else {}
        for doc in col.find(filtro).sort("_id", 1):
            ultimo_id = doc["_id"]
            for t, cmd in procesar_doc(doc):
                eventos.append((t, cmd)); ultimo_cmd = (cmd, time.monotonic())
        if time.monotonic() - t_dib >= REFRESCO:
            dibujar(ultimo_cmd); t_dib = time.monotonic()
        time.sleep(SONDEO)
except KeyboardInterrupt:
    pass
print(f"⏹ Fin · {len(eventos)} comandos detectados")

In [ ]:
if len(hist) == 0:
    print("Aún no hay datos calibrados para graficar.")
else:
    h = np.array(hist)                                  # columnas: t, pitch, roll
    corte = np.where(np.diff(h[:, 0]) < -1)[0]          # si el reloj se reinició, solo la última sesión
    h = h[corte[-1] + 1:] if corte.size else h
    ev = [(t, c) for t, c in eventos if t >= h[0, 0]]
    lim = max(80, np.abs(h[:, 1:]).max() * 1.5)
    fig, ax = plt.subplots(figsize=(11, 4.4))
    ax.plot(h[:, 0], h[:, 1], color=AZUL, lw=1.8, label="↕ pitch (gy)")
    ax.plot(h[:, 0], h[:, 2], color=NARANJA, lw=1.8, label="↔ roll (gx)")
    for y, ls in ((TH_ON, "--"), (-TH_ON, "--"), (TH_OFF, ":"), (-TH_OFF, ":")):
        ax.axhline(y, color=GRIS, ls=ls, lw=1)
    ax.text(h[-1, 0], TH_ON + 1.5, "TH_ON = ±25°", ha="right", color=GRIS, fontsize=9)
    ax.text(h[-1, 0], TH_OFF + 1.5, "TH_OFF = ±10°", ha="right", color=GRIS, fontsize=9)
    for t, cmd in ev:
        ax.axvline(t, color=ROJO, alpha=0.4, lw=1.3)
        ax.text(t + 0.08, lim * 0.95, cmd, rotation=90, va="top", color=ROJO, fontsize=10, fontweight="bold")
    ax.set(ylim=(-lim, lim), xlabel="t  (s)", ylabel="ángulo θ  (grados)",
           title="Ángulo calculado con tus datos reales y comandos detectados")
    ax.legend(loc="lower left"); plt.tight_layout(); plt.show()

**Qué muestra la gráfica**

- Cada gesto hace subir el ángulo hasta cruzar **±25°**: ahí se dispara el comando (línea roja).
- Mientras el ángulo no baje de **±10°**, el comando queda bloqueado: no hay repeticiones.
- Al volver a la posición inicial, el mando se rearma y queda listo para el siguiente gesto.

## 8. 🖥️ Ahora con el juego

Lo que acabamos de ver pasa dentro del programa que corre en mi computador, con una diferencia: **pulsa las teclas**. Corro tres cosas:

1. **`main.py`** → recibe los datos del celular y los guarda en MongoDB.
2. **`python mando_local_todo.py`** → calcula el ángulo y **pulsa las teclas**.
3. **El juego** en el navegador (con clic en su ventana para que reciba las flechas).

**Antes de empezar**

☐ Celular y PC en la misma red, con Phyphox apuntando a la IP del PC.
☐ Dar *play* en Phyphox y **no mover el celular ~2 s** (calibración).
☐ Esperar el mensaje 🟢 LISTO.
☐ Comprobar que `lote llegó hace` sea de menos de ~1 s.
☐ El gesto es **inclinar y volver rápido**. Si mantengo la inclinación más de ~0.3 s, el mando re-centra el ángulo ahí y el regreso cuenta como el comando contrario.

## 9. ✅ Conclusiones

- 🧭 El problema es una **EDO de primer orden con valor inicial**: $\theta' = \omega(t)$, $\theta(0)=0$.
- 🧮 Probamos **Euler** y **Heun**. Como $f$ no depende de θ, Heun se reduce a la **regla del trapecio**: es de **orden 2** y no cuesta nada extra.
- 🧹 Lo que hace funcionar el mando no es solo el método: son la **calibración, la zona muerta, el re-centrado y la histéresis**.

### Limitaciones
- Integrar acumula error (deriva): lo controlamos, pero no lo eliminamos.
- Tratamos cada eje por separado (válido para giros pequeños o en un solo eje).
- Solo usamos dos ejes (gx, gy).

### Ideas a futuro
Usar también el acelerómetro (filtro complementario), añadir el tercer eje y calibrar de forma continua.